# Chapter 12 assignment: PINNs for dynamic equations on time scales

Companion notebook to Section *Dynamic equations on time scales* of Chapter 12 (Further equation
classes). It runs on a CPU in a few minutes; no GPU is needed.

**Learning goals**

1. Compute the forward jump $\sigma$ and the graininess $\mu$ of a time scale given as a union of
   intervals and points.
2. Assemble the PINN loss that uses automatic differentiation at right-dense points and forward
   differences at right-scattered points.
3. Use one code for continuous time ($\mathbb{R}$), discrete time ($\mathbb{Z}$) and hybrid time
   ($\mathbb{P}_{1,1}=\bigcup_k[2k,2k+1]$), and compare with the exact time-scale exponential.
4. See why a single analytic network cannot be exact across the gaps of a time scale, and how one
   output per connected component removes the obstruction.
5. Observe the sign changes of $e_p$ when $1+\mu p<0$ and the convergence $h\mathbb{Z}\to\mathbb{R}$.

## Short theory recap

* A **time scale** $\mathbb{T}$ is a nonempty closed subset of $\mathbb{R}$; $\sigma(t)=\inf\{s\in\mathbb{T}:s>t\}$,
  $\mu(t)=\sigma(t)-t$. A point is right-dense if $\mu(t)=0$ and right-scattered if $\mu(t)>0$.
* **Delta derivative:** $y^\Delta(t)=y'(t)$ at right-dense points and $y^\Delta(t)=(y(\sigma(t))-y(t))/\mu(t)$
  at right-scattered points.
* $y^\Delta=p\,y$, $y(0)=1$, with $1+\mu p\neq0$ (regressivity) has the solution
  $e_p(t,0)=\exp\bigl(p\,|\mathbb{T}\cap[0,t]|\bigr)\prod_{s<t,\ \mu(s)>0}(1+\mu(s)p)$ for constant $p$;
  on $\mathbb{P}_{1,1}$: $e_p(t,0)=((1+p)e^p)^k e^{p(t-2k)}$ for $t\in[2k,2k+1]$.
* **PINN loss:**
  $\mathcal{L}=\frac{1}{M_d}\sum_{\text{right-dense}}(y_\theta'-py_\theta)^2
  +\frac{1}{M_s}\sum_{\text{right-scattered}}\bigl(\frac{y_\theta(\sigma)-y_\theta}{\mu}-py_\theta\bigr)^2$;
  the two groups are averaged separately so that the few jump conditions are not drowned.

In [ ]:
import numpy as np
import torch

from pinnode import MLP, configure_torch, derivative, rel_l2, set_seed, to_tensor, train

configure_torch("float64", threads=1)   # double precision on one CPU thread, as in the book
set_seed(1234)

R7 = [(0.0, 7.0)]                              # the continuum [0, 7]
Z8 = [(float(k), float(k)) for k in range(8)]  # the integers 0, 1, ..., 7
P11 = [(2.0 * k, 2.0 * k + 1.0) for k in range(4)]   # [0,1] U [2,3] U [4,5] U [6,7]


## Task 1: forward jump, graininess and collocation points

Complete the two functions. Components are pairs `(a, b)` with `a <= b`; `a == b` is an isolated point.

In [ ]:
def scattered_points(components):
    """Right-scattered points of T = U_c [a_c, b_c] (components sorted and disjoint).

    Returns three arrays (t, sigma, mu): the points b_c of all components except the last one,
    their forward jumps sigma(b_c) = a_{c+1} and graininess mu = a_{c+1} - b_c.
    """
    # TODO: loop over consecutive pairs of components
    raise NotImplementedError


def dense_points(components, n_per_unit=50):
    """Uniform right-dense collocation points in the nondegenerate components.

    The right end point b_c of every component except the last is right-scattered and must be
    excluded; max T is right-dense by convention (sigma(max T) = max T).
    """
    # TODO: np.linspace(a, b, n + 1) with n = round(n_per_unit * (b - a)); drop b unless last
    raise NotImplementedError


**Check your answer.**

In [ ]:
t, s, m = scattered_points(P11)
assert np.allclose(t, [1, 3, 5]) and np.allclose(s, [2, 4, 6]) and np.allclose(m, 1)
t, s, m = scattered_points(Z8)
assert np.allclose(t, np.arange(7)) and np.allclose(m, 1)
assert len(scattered_points(R7)[0]) == 0
d = dense_points(P11, 50)
assert np.isclose(d.min(), 0) and np.isclose(d.max(), 7) and not np.any(np.isclose(d, 1.0))
assert dense_points(Z8).size == 0
print("Task 1: correct")


## Task 2: the exact time-scale exponential

In [ ]:
def exp_p(t, p, components):
    """Time-scale exponential e_p(t, 0) for constant p and t in T.

    e_p(t, 0) = exp(p * |T cap [0, t]|) * prod_{right-scattered s < t} (1 + mu(s) p)
    """
    t = np.atleast_1d(np.asarray(t, float))
    out = np.empty_like(t)
    # TODO: for every t_j, add up the lengths of the parts of the intervals left of t_j and
    #       multiply the factors (1 + mu p) of the right-scattered points s < t_j
    raise NotImplementedError
    return out


**Check your answer.**

In [ ]:
p = 0.5
assert np.isclose(exp_p(7.0, p, R7)[0], np.exp(7 * p))
assert np.isclose(exp_p(7.0, p, Z8)[0], (1 + p) ** 7)
assert np.isclose(exp_p(7.0, p, P11)[0], ((1 + p) * np.exp(p)) ** 3 * np.exp(p))
assert np.isclose(exp_p(2.5, p, P11)[0], (1 + p) * np.exp(p) * np.exp(0.5 * p))
print("Task 2: correct")


## Task 3: the PINN loss on a time scale

The model below is complete: it evaluates the output belonging to the component of each point. Complete the residuals in `make_loss`.

In [ ]:
class TSModel(torch.nn.Module):
    """y(t) = 1 + t N_c(t): shared hidden layers, one output per component c (or a single output)."""

    def __init__(self, components, heads=True, width=32, depth=3, T=7.0):
        super().__init__()
        self.starts = np.array([a for a, _ in components])
        self.heads = heads
        self.net = MLP(1, len(components) if heads else 1, width=width, depth=depth,
                       activation="tanh", domain=(0.0, T))

    def comp(self, t_np):
        """Index of the component containing each point (points must lie in T)."""
        idx = np.searchsorted(self.starts, np.asarray(t_np, float) + 1e-12, side="right") - 1
        return torch.as_tensor(idx, dtype=torch.long).reshape(-1, 1)

    def forward(self, t, idx):
        n = self.net(t)
        if self.heads:
            n = n.gather(1, idx)
        return 1.0 + t * n          # y(0) = 1 exactly


def test_grid(components, n=1001):
    """Uniform points on the intervals of T plus every isolated point."""
    length = sum(b - a for a, b in components)
    pts = [np.linspace(a, b, max(2, int(round(n * (b - a) / length)))) if b > a else np.array([a])
           for a, b in components]
    return np.concatenate(pts)


def solve(components, p, heads=True, adam=3000, lbfgs=300, seed=1234):
    """Train the PINN and return (t_test, prediction, exact, relative L2 error)."""
    torch.manual_seed(seed)
    model = TSModel(components, heads)
    loss_fn = make_loss(model, components, p)
    train(model.parameters(), loss_fn, adam_iters=adam, lr=1e-3, lr_final=1e-5,
          lbfgs_iters=lbfgs, print_every=0, verbose=False)
    t = test_grid(components)
    with torch.no_grad():
        y = model(to_tensor(t), model.comp(t)).numpy().ravel()
    ex = exp_p(t, p, components)
    return t, y, ex, rel_l2(y, ex)


In [ ]:
def make_loss(model, components, p, n_per_unit=50):
    """Time-scale PINN loss: mean squared right-dense residual + mean squared right-scattered residual."""
    td_np = dense_points(components, n_per_unit)
    ts_np, sig_np, mu_np = scattered_points(components)
    td = to_tensor(td_np, requires_grad=True) if td_np.size else None
    id_ = model.comp(td_np) if td_np.size else None
    ts, sig, mu = to_tensor(ts_np), to_tensor(sig_np), to_tensor(mu_np)
    i_ts, i_sig = model.comp(ts_np), model.comp(sig_np)

    def loss_fn():
        parts, loss = {}, 0.0
        if td is not None:
            # TODO: right-dense residual  y' - p y  by automatic differentiation (pinnode.derivative)
            raise NotImplementedError
        if len(ts_np):
            # TODO: right-scattered residual  (y(sigma) - y(t)) / mu - p y(t)
            raise NotImplementedError
        return loss, parts
    return loss_fn


## Task 4: one code, three time scales

Train on $\mathbb{R}$, $\mathbb{Z}$ and $\mathbb{P}_{1,1}$ with $p=1/2$ (about a minute in total).

In [ ]:
errors = {}
for name, T in (("R", R7), ("Z", Z8), ("P11", P11)):
    t, y, ex, err = solve(T, 0.5, heads=True)
    errors[name] = err
    print(f"{name:4s} relative L2 error = {err:.2e}")


**Check your answer.**

In [ ]:
assert all(e < 1e-3 for e in errors.values()), errors
print("Task 4: correct")


## Task 5: a single output versus one head per interval

A $\tanh$ network is real-analytic. If it equals $e^{pt}$ on $[0,1]$, what is its value at $t=2$? Compare with $e_p(2,0)=(1+p)e^p$.

In [ ]:
t, y_single, ex, err_single = solve(P11, 0.5, heads=False)
t, y_heads, ex, err_heads = solve(P11, 0.5, heads=True)
print(f"single output: {err_single:.2e}   one head per interval: {err_heads:.2e}")
# TODO: explain the difference in two sentences (Proposition on real-analytic networks)


**Check your answer.**

In [ ]:
assert err_heads < err_single
print("Task 5: correct -- the heads remove the obstruction of a single analytic output")


## Task 6: sign changes for $1+\mu p<0$

For $p=-1.5$ the factor $1+p$ is negative (see the exercise on exponential functions on time scales in Chapter 12).

In [ ]:
p = -1.5
t, y, ex, err = solve(P11, p, heads=True)
print(f"p = {p}: relative L2 error = {err:.2e}")
# TODO: plot y and ex on each interval (matplotlib) and mark the sign changes


**Check your answer.**

In [ ]:
assert np.all(np.sign(y) == np.sign(ex)), "the sign pattern must match the exact exponential"
assert err < 1e-2
print("Task 6: correct")


## Task 7: from $h\mathbb{Z}$ to $\mathbb{R}$

On $h\mathbb{Z}$ the exponential is $(1+hp)^{t/h}$. Optional: train the PINN on $h\mathbb{Z}$ for $h=0.25$ and compare.

In [ ]:
p = 0.5
for h in (1.0, 0.5, 0.25, 0.125, 0.0625):
    Th = [(k * h, k * h) for k in range(int(round(7 / h)) + 1)]
    # TODO: compare exp_p(7, p, Th) with exp(7 p) and with the closed form (1 + h p)^(7/h)


**Check your answer.**

In [ ]:
vals = [exp_p(7.0, 0.5, [(k * h, k * h) for k in range(int(round(7 / h)) + 1)])[0]
        for h in (1.0, 0.5, 0.25, 0.125, 0.0625)]
gaps = np.abs(np.array(vals) - np.exp(3.5))
assert np.all(np.diff(gaps) < 0) and np.isclose(vals[0], 1.5 ** 7)
print("Task 7: correct -- e_p on hZ converges to exp(pt) as h -> 0")


## Going further

* Replace $p$ by $p(t)=\cos(\pi t)$ on $\mathbb{P}_{1,1}$ and derive the exact solution.
* Add a delay: $y^\Delta(t)=p\,y(t-2)$ on $\mathbb{P}_{1,1}$ (the shift by 2 maps $\mathbb{P}_{1,1}$ into itself) with history $y=1$ on $[-2,-1]$; delay equations on time scales are treated in Georgiev, *Functional Dynamic Equations on Time Scales* (Springer, 2019).